# Keypoint Combining MediaPipe & MoveNet

MediaPipe is used as the base model.
1. MediaPipe visibility ≥ 0.6 → use MediaPipe
2. Else MoveNet confidence ≥ 0.6 and not interpolated → use MoveNet
3. Else → NaN

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import re

IN_DIR    = Path('../data/processed/keypoints_interpolated_boundary')
VIDEO_DIR = Path('../data/original/Utvalda filminspelningar för IRAF analys/Dec 2025 sit-stå och stå-sitt')
OUT_CSV   = Path('../data/processed/keypoints_combined')
OUT_VIDEO = Path('../data/processed/keypoints_combined_videos')

FPS       = 50
THRESHOLD = 0.6

time_dict = {
    '028': (9.90,  11.77), '030': (5.80,  7.63),
    '045': (6.54,   7.98), '047': (7.03,  8.91),
    '059': (6.65,   9.11), '061': (6.00,  9.00),
    '074': (6.07,   7.27), '076': (5.00,  8.00),
    '091': (6.39,   7.61), '093': (5.86,  7.67),
}

JOINTS = [
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

COLOR_MEDIAPIPE = (255, 80,  80)   # blue
COLOR_MOVENET   = (80,  200, 80)   # green

OUT_CSV.mkdir(parents=True, exist_ok=True)
OUT_VIDEO.mkdir(parents=True, exist_ok=True)

## Combining function

In [6]:
def combine_keypoints(df_mp, df_mn):
    combined = df_mp.copy()

    for joint in JOINTS:
        mp_vis    = df_mp[f'{joint}_visibility'].values
        mn_conf   = df_mn[f'{joint}_confidence'].values
        mn_interp = df_mn[f'{joint}_interpolated'].values if f'{joint}_interpolated' in df_mn.columns else np.ones(len(df_mn), dtype=bool)

        mp_missing  = mp_vis < THRESHOLD
        mn_good     = (mn_conf >= THRESHOLD) & (~mn_interp.astype(bool))
        use_movenet = mp_missing & mn_good

        for coord in ['x', 'y']:
            combined.loc[use_movenet, f'{joint}_{coord}'] = df_mn.loc[use_movenet, f'{joint}_{coord}'].values

        combined[f'{joint}_source'] = 'mediapipe'
        combined.loc[use_movenet, f'{joint}_source'] = 'movenet'
        combined.loc[mp_missing & ~mn_good, f'{joint}_source'] = 'missing'

        still_missing = mp_missing & ~mn_good
        combined.loc[still_missing, f'{joint}_x'] = np.nan
        combined.loc[still_missing, f'{joint}_y'] = np.nan

    return combined

## Apply and save combined CSVs

In [7]:
for mp_path in sorted((IN_DIR / 'mediapipe_norm').glob('*.csv')):
    video_id = mp_path.stem.replace('_mediapipe_norm', '')
    mn_path  = IN_DIR / 'movenet' / f'{video_id}_movenet.csv'

    if not mn_path.exists():
        print(f'  no MoveNet match for {video_id}, skipping')
        continue

    df_mp = pd.read_csv(mp_path)
    df_mn = pd.read_csv(mn_path)

    n          = min(len(df_mp), len(df_mn))
    df_combined = combine_keypoints(df_mp.iloc[:n].reset_index(drop=True),
                                    df_mn.iloc[:n].reset_index(drop=True))

    df_combined.to_csv(OUT_CSV / f'{video_id}.csv', index=False)
    print(f'{video_id}: {n} frames saved')

DJI_20250425092743_0028_D: 94 frames saved
DJI_20250425093100_0030_D: 92 frames saved


DJI_20250425104507_0045_D: 73 frames saved
DJI_20250425104804_0047_D: 94 frames saved
DJI_20250425112502_0059_D: 123 frames saved
DJI_20250425112749_0061_D: 151 frames saved
DJI_20250425120835_0074_D: 60 frames saved
DJI_20250425121226_0076_D: 151 frames saved
DJI_20250425125202_0091_D: 61 frames saved
DJI_20250425125448_0093_D: 91 frames saved


## Generate combined videos

- Blue = MediaPipe keypoint
- Green = MoveNet fill-in

In [8]:
def draw_joints(frame, row, h, w):
    for joint in JOINTS:
        x = row.get(f'{joint}_x')
        y = row.get(f'{joint}_y')
        if pd.isna(x) or pd.isna(y):
            continue
        color = COLOR_MOVENET if row.get(f'{joint}_source') == 'movenet' else COLOR_MEDIAPIPE
        cv2.circle(frame, (int(x * w), int(y * h)), 7, color, -1)
        cv2.circle(frame, (int(x * w), int(y * h)), 7, (0, 0, 0), 1)
    return frame


for csv_path in sorted(OUT_CSV.glob('*.csv')):
    match = re.search(r'_0(\d{3})_D', csv_path.stem)
    if not match:
        continue
    vid_id    = match.group(1)
    vid_files = list(VIDEO_DIR.glob(f'*_0{vid_id}_D.MP4'))

    if not vid_files:
        print(f'Video not found: {vid_id}')
        continue

    df  = pd.read_csv(csv_path)
    cap = cv2.VideoCapture(str(vid_files[0]))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(time_dict[vid_id][0] * FPS))

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_path = OUT_VIDEO / f'{csv_path.stem}.mp4'
    writer   = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'avc1'), FPS, (w, h))

    for i, row in df.iterrows():
        ret, frame = cap.read()
        if not ret:
            break
        frame = draw_joints(frame, row, h, w)
        writer.write(frame)

    cap.release()
    writer.release()
    print(f'{vid_id}: video saved')

028: video saved
030: video saved
045: video saved
047: video saved
059: video saved
061: video saved
074: video saved
076: video saved
091: video saved
093: video saved
